# Dijkstra

In [ ]:
!pip install networkx matplotlib
import networkx as nx
import matplotlib.pyplot as plt

def get_min_dist(dist, visited, v):
    min_dist = float('inf')
    min_index = -1
    for i in range(v): # Fixed: changed variable 'v' to 'i' to avoid shadowing the length parameter
        if dist[i] < min_dist and not visited[i]:
            min_dist = dist[i]
            min_index = i
    return min_index

def dijkstra(graph, src):
    v = len(graph)
    distances = [float('inf')] * v
    distances[src] = 0
    visited = [False] * v
    previous = [-1] * v

    for _ in range(v):
        u = get_min_dist(distances, visited, v)
        if u == -1: # Added safety check for disconnected graphs
            break 
        visited[u] = True

        for neighbor in range(v):
            edge_weight = graph[u][neighbor]
            if edge_weight > 0 and not visited[neighbor]:
                new_distance = distances[u] + edge_weight
                if new_distance < distances[neighbor]:
                    distances[neighbor] = new_distance
                    previous[neighbor] = u

    return distances, previous

def get_shortest_path(previous, src, dest):
    path = []
    current = dest
    while current != -1:
        path.append(current)
        current = previous[current]
    path.reverse()

    # Verify if a valid path exists
    if len(path) > 0 and path[0] == src:
        print("Shortest path from {} to {}: {}".format(src, dest, " -> ".join(map(str, path))))
        return path
    else:
        print("No path exists from {} to {}".format(src, dest))
        return []

def plot_graph(graph_matrix, shortest_path=None):
    G = nx.Graph()
    v = len(graph_matrix)
    
    # Add nodes and edges from your adjacency matrix
    for i in range(v):
        for j in range(i + 1, v): # Start from i+1 for undirected graph to avoid duplicates
            weight = graph_matrix[i][j]
            if weight > 0:
                G.add_edge(i, j, weight=weight)

    # Set up the layout
    pos = nx.spring_layout(G, seed=42) # Seed ensures the graph looks the same every time
    
    plt.figure(figsize=(8, 6))

    # 1. Draw the base graph (nodes and standard edges)
    nx.draw_networkx_nodes(G, pos, node_color='lightgray', node_size=700)
    nx.draw_networkx_edges(G, pos, width=2, alpha=0.5, edge_color='gray')
    nx.draw_networkx_labels(G, pos, font_size=14, font_family='sans-serif', font_weight='bold')

    # 2. Add the edge weights (labels)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12)

    # 3. Highlight the shortest path if one was found
    if shortest_path and len(shortest_path) > 1:
        # Highlight nodes in the path
        nx.draw_networkx_nodes(G, pos, nodelist=shortest_path, node_color='lightgreen', node_size=700)
        
        # Create a list of edge tuples from the path list (e.g., [0, 1, 2] -> [(0,1), (1,2)])
        path_edges = list(zip(shortest_path, shortest_path[1:]))
        
        # Highlight edges in the path
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='red', width=4)
        plt.title(f"Shortest Path highlighted in Red", fontsize=16)
    else:
        plt.title("Graph Visualization", fontsize=16)

    plt.axis('off') # Hide the grid axes
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    graph = [
        [0, 7, 9, 0, 0, 14],
        [7, 0, 10, 15, 0, 0],
        [9, 10, 0, 11, 0, 2],
        [0, 15, 11, 0, 6, 0],
        [0, 0, 0, 6, 0, 9],
        [14, 0, 2, 0, 9, 0]
    ]

    src = int(input("Enter the source vertex (0-5): "))
    dest = int(input("Enter the destination vertex (0-5): "))

    distances, previous = dijkstra(graph, src)
    
    # Capture the path list so we can pass it to the plotter
    path = get_shortest_path(previous, src, dest)
    
    # Plot the results
    plot_graph(graph, path)

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
Shortest path from 1 to 4: 1 -> 2 -> 5 -> 4


# A* Algorithm

In [3]:
def heuristic(u, v, coords):
    x1, y1 = coords[u]
    x2, y2 = coords[v]
    return abs(x1 - x2) + abs(y1 - y2)

def get_min_f_score(f_score, visited, V):

    minimum = float('inf')
    min_index = -1
    
    for v in range(V):
        if f_score[v] < minimum and not visited[v]:
            minimum = f_score[v]
            min_index = v
            
    return min_index

def a_star(graph, start, goal, coords):
    V = len(graph)

    g_score = [float('inf')] * V
    g_score[start] = 0
    
    f_score = [float('inf')] * V
    f_score[start] = heuristic(start, goal, coords)
    
    visited = [False] * V
    previous = [-1] * V

    for _ in range(V):
        u = get_min_f_score(f_score, visited, V)
        
        if u == goal or u == -1:
            break
        visited[u] = True

        for v in range(V):
            edge_weight = graph[u][v]
            
            if edge_weight > 0 and not visited[v]:
                tentative_g = g_score[u] + edge_weight
                
                if tentative_g < g_score[v]:
                    previous[v] = u
                    g_score[v] = tentative_g
                    f_score[v] = tentative_g + heuristic(v, goal, coords)

    return g_score, previous
if __name__ == "__main__":
    graph = [
        [0, 7, 9, 0, 0, 14],
        [7, 0, 10, 15, 0, 0],
        [9, 10, 0, 11, 0, 2],
        [0, 15, 11, 0, 6, 0],
        [0, 0, 0, 6, 0, 9],
        [14, 0, 2, 0, 9, 0]
    ]

    coords = [
        (0, 0),   # Node A
        (1, 2),   # Node B
        (2, 1),   # Node C
        (3, 3),   # Node D
        (4, 2),   # Node E
        (5, 1)    # Node F
    ]

    start = int(input("Enter the start vertex (0-5): "))
    goal = int(input("Enter the goal vertex (0-5): "))

    g_score, previous = a_star(graph, start, goal, coords)
    print_shortest_path(previous, start, goal)

Shortest path from 1 to 5: 1 -> 2 -> 5
